<a href="https://colab.research.google.com/github/Lariuki/Lariuki/blob/main/Simple_ReAct_Agent_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install python-dotenv

In [2]:
!pip install openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.6/328.6 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.8 MB/s eta 0:00:00


In [5]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = "sk-regazzo-ia-poc-lab-Iuj8XTwW7o9ZWebsNW3LT3BlbkFJWYXtCVujTQCAhWxG5MhQ"

In [6]:
client = OpenAI()

In [7]:
chat_completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Hello world"}]
)

In [8]:
chat_completion.choices[0].message.content

'Hello! How are you today?'

In [9]:
# Criou uma classe para o agente, fazer com que o agente salve isso e salve como um atributo
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gpt-4o",
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content

In [27]:
#Estamos pedindo que ele percorra esse ciclo de pensamento, ação, pausa e observação.
#Eu poderia então produzir uma resposta quando terminar com isso. Ele usará o pensamento.
#Em seguida, ele usará a ação para executar uma das opções disponíveis.
#E então retornará uma pausa após essa observação ser usada para sinalizar o resultado da execução dessas ações.
#Em seguida, informamos quais são as ações disponíveis.
#Portanto, damos-lhe acesso para calcular o peso médio do cão e iremos implementá-lo abaixo.
#Finalmente, fornecemos um exemplo disso em ação neste exemplo.
prompt = """
Você corre em um ciclo de Pensamento, Ação, PAUSA, Observação.
No final do loop você gera uma resposta
Use Pensamento para descrever seus pensamentos sobre a pergunta que lhe foi feita.
Use Action para executar uma das ações disponíveis para você - depois retorne PAUSE.
A observação será o resultado da execução dessas ações.

Suas ações disponíveis são:

calcular:
por exemplo. calcular: 4 * 7/3
Executa um cálculo e retorna o número - usa Python, portanto, certifique-se de usar a sintaxe de ponto flutuante, se necessário

peso_médio_dog:
por exemplo. peso_médio_dog: Collie
retorna o peso médio de um cachorro quando dada a raça

Sessão de exemplo:

Pergunta: Quanto pesa um Bulldog?
Pensei: eu deveria olhar o peso dos cães usando Average_dog_weight
Ação: Average_dog_weight: Bulldog
PAUSA

Você será chamado novamente com isto:

Observação: um Bulldog pesa 51 libras

Você então produz:

Resposta: Um bulldog pesa 51 libras
""".strip()

In [28]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier":
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [29]:
abot = Agent(prompt)

In [30]:
result = abot("Quanto pesa um poodle toy?")
print(result)

Pensei: Eu deveria olhar o peso dos cães usando peso_médio_dog.
Ação: peso_médio_dog: Poodle Toy
PAUSA


In [31]:
result = average_dog_weight("Toy Poodle")

In [32]:
result

'a toy poodles average weight is 7 lbs'

In [33]:
next_prompt = "Observation: {}".format(result)

In [34]:
abot(next_prompt)

'Resposta: Um Poodle Toy pesa em média 7 libras.'

In [35]:
abot.messages

[{'role': 'system',
  'content': 'Você corre em um ciclo de Pensamento, Ação, PAUSA, Observação.\nNo final do loop você gera uma resposta\nUse Pensamento para descrever seus pensamentos sobre a pergunta que lhe foi feita.\nUse Action para executar uma das ações disponíveis para você - depois retorne PAUSE.\nA observação será o resultado da execução dessas ações.\n\nSuas ações disponíveis são:\n\ncalcular:\npor exemplo. calcular: 4 * 7/3\nExecuta um cálculo e retorna o número - usa Python, portanto, certifique-se de usar a sintaxe de ponto flutuante, se necessário\n\npeso_médio_dog:\npor exemplo. peso_médio_dog: Collie\nretorna o peso médio de um cachorro quando dada a raça\n\nSessão de exemplo:\n\nPergunta: Quanto pesa um Bulldog?\nPensei: eu deveria olhar o peso dos cães usando Average_dog_weight\nAção: Average_dog_weight: Bulldog\nPAUSA\n\nVocê será chamado novamente com isto:\n\nObservação: um Bulldog pesa 51 libras\n\nVocê então produz:\n\nResposta: Um bulldog pesa 51 libras'},
 {'

In [36]:
abot = Agent(prompt)

In [37]:
question = """Tenho 2 cachorros, um border collie e um terrier escocês. \
Qual é o peso combinado deles"""
abot(question)

'Penso que devo descobrir o peso médio de cada uma das raças de cães mencionadas e, em seguida, somar esses pesos para obter o peso combinado.\n\nAção: peso_médio_dog: Border Collie\nPAUSA'

In [38]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

Observation: a Border Collies average weight is 37 lbs


In [39]:
abot(next_prompt)

'Penso que agora devo descobrir o peso médio de um Terrier Escocês.\n\nAção: peso_médio_dog: Terrier Escocês\nPAUSA'

In [40]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

Observation: Scottish Terriers average 20 lbs


In [41]:
abot(next_prompt)

'Penso que agora devo somar os pesos médios do Border Collie e do Terrier Escocês para obter o peso combinado.\n\nAção: calcular: 37 + 20\nPAUSA'

In [42]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

Observation: 57


In [43]:
abot(next_prompt)

'Resposta: O peso combinado de um Border Collie e um Terrier Escocês é de 57 libras.'

Add loop

In [44]:
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

In [45]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

In [46]:
question = """Tenho 2 cachorros, um border collie e um terrier escocês. \
Qual é o peso combinado deles"""
query(question)

Penso que devo determinar o peso médio de cada raça de cachorro e, em seguida, somar esses valores para obter o peso combinado.

Ação: peso_médio_dog: Border Collie
PAUSA
